# A/B Testing & Hypothesis Testing — Checkout Experiment

This notebook analyzes `checkout_ab_test_v2.csv` using a two-proportion Z-test and chi-square test.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm, chi2_contingency

df = pd.read_csv('checkout_ab_test_v2.csv')
df.head()

## 1. Data validation

In [ ]:
print('Shape:', df.shape)
print('Missing values:\n', df.isna().sum())
print('Variants:', df['Variant'].value_counts().to_dict())
print('Conversion labels:', df['Converted'].value_counts().to_dict())

## 2. Hypotheses

**H₀:** pB − pA = 0

**H₁:** pB − pA ≠ 0

Use α = 0.05 and a two-sided two-proportion Z-test.

In [ ]:
df['converted_flag'] = (df['Converted'].str.lower() == 'yes').astype(int)
summary = df.groupby('Variant').agg(n=('converted_flag','size'), conversions=('converted_flag','sum'))
summary['conversion_rate'] = summary['conversions']/summary['n']
summary

## 3. Conversion lift and confidence interval

In [ ]:
a = summary.loc['A']; b = summary.loc['B']
pA, pB = a.conversion_rate, b.conversion_rate
diff = pB - pA
relative_lift = diff / pA
se_diff = np.sqrt(pA*(1-pA)/a.n + pB*(1-pB)/b.n)
ci = (diff - 1.96*se_diff, diff + 1.96*se_diff)
print(f'Control rate: {pA:.2%}')
print(f'Variant rate: {pB:.2%}')
print(f'Absolute lift: {diff:.2%}')
print(f'Relative lift: {relative_lift:.1%}')
print(f'95% CI for B-A: [{ci[0]:.2%}, {ci[1]:.2%}]')

## 4. Two-proportion Z-test

In [ ]:
pooled = (a.conversions+b.conversions)/(a.n+b.n)
se = np.sqrt(pooled*(1-pooled)*(1/a.n + 1/b.n))
z = diff/se
p_value = 2*norm.sf(abs(z))
print(f'Z statistic: {z:.4f}')
print(f'p-value: {p_value:.4f}')
print('Decision:', 'Reject H0' if p_value < 0.05 else 'Fail to reject H0')

## 5. Chi-square test

In [ ]:
table = [[a.conversions, a.n-a.conversions], [b.conversions, b.n-b.conversions]]
chi2, p_chi, dof, expected = chi2_contingency(table, correction=False)
print(f'Chi-square: {chi2:.4f}')
print(f'p-value: {p_chi:.4f}')

## 6. Executive decision memo

Variant B has an observed positive conversion lift, but the two-sided p-value is above 0.05 and the 95% confidence interval for B − A includes zero. Under the stated 5% significance rule, retain Control A rather than treating Variant B as a validated improvement.

## 7. Reproducibility note

The achieved sample size is 600 visitors per arm. A required pre-test sample size would require a baseline conversion rate, minimum detectable effect, power target, and significance level; these were not specified in the assignment dataset.